# #01 **From `Dense` to `MultiDense`**

----

## 1. Imports

In [ ]:
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense

In [ ]:
from multi_dense import MultiDense

----

## 2. Ensure experimental reproductibility

In [ ]:
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

----

## 3. Models Definitions

Each model will be composed of **four layers** (one input layer, two hidden layers, and one output layer) for a total of three computational layers.

These layers are configured in the following manner:

| Index | Type   | Num. of Neurons | Activation Functions |
|-------|--------|-----------------|----------------------|
| 0     | Input  | 784             |                      |
| 1     | Hidden | 128             | ReLU + Sigmoid       |
| 2     | Hidden | 64              | ReLU + Sigmoid       |
| 3     | Output | 10              | Softmax              |


In [ ]:
models: dict[str, Sequential] = dict()

### 3.1. Models

#### 3.1.1. M: **100+0**
100% ReLU + 0% Sigmoid neuron activations per hidden layer.

In [ ]:
models["100+0"] = Sequential(
    [
        MultiDense([128], ["relu"]),
        MultiDense([64], ["relu"]),
        Dense(10, "softmax"),
    ]
)

#### 3.1.2. M: **75+25**
75% ReLU + 35% Sigmoid neuron activations per hidden layer.<>

In [ ]:
models["75+25"] = Sequential(
    [
        MultiDense([96, 32], ["relu", "sigmoid"]),
        MultiDense([48, 16], ["relu", "sigmoid"]),
        Dense(10, "softmax"),
    ]
)

#### 3.1.3. M: **50+50**
50% ReLU + 50% Sigmoid neuron activations per hidden layer.

In [ ]:
models["50+50"] = Sequential(
    [
        MultiDense([64, 64], ["relu", "relu"] ),
        MultiDense([32, 32], ["relu", "relu"]),
        Dense(10, "softmax"),
    ]
)

#### 3.1.4. M: **25+75**
25% ReLU + 75% Sigmoid neuron activations per hidden layer.

In [ ]:
models["25+75"] = Sequential(
    [
        MultiDense([32, 96], ["relu", "sigmoid"]),
        MultiDense([16, 48], ["relu", "sigmoid"]),
        Dense(10, "softmax"),
    ]
)

#### 3.1.5. M: **0+100**
0% ReLU + 100% Sigmoid neuron activations per hidden layer.

In [ ]:
models["0+100"] = Sequential(
    [
        MultiDense([128], ["sigmoid"]),
        MultiDense([64], ["sigmoid"]),
        Dense(10, "softmax"),
    ]
)

### 3.2. Models Compilation

In [ ]:
for tag, model in models.items():
    print(f"Compiling {tag} model...")

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print(f"{tag} model compiled!")
    print()

----

## 4. Experimentation

### 4.1. Load

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

### 4.2. Preprocessing

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 28**2)
x_test = x_test.reshape(-1, 28**2)

### 4.3. Training

In [ ]:
for tag, model in models.items():
    print(f"Training {tag} model...")

    model.fit(
        x_train,
        y_train,
        epochs=10,
        batch_size=100,
        validation_split=0.1,
        verbose=0,
    )

    print(f"{tag} model trained!")
    print()

### 4.2. Testing

In [ ]:
models_evaluation = {
    tag: model.evaluate(x_test, y_test) for tag, model in models.items()
}

----

### 5. Conclusions

In [ ]:
for tag, (loss, accuracy) in models_evaluation.items():
    print(f"{tag} model (Loss: {loss:.4}, Accuracy: {accuracy:.2%}).")

----